In [1]:
%load_ext autoreload
%autoreload 2

import torch

In [2]:
from tqdm.notebook import tqdm_notebook as tqdm
import time

total_epochs = 100
epoch_bar = tqdm(range(total_epochs), desc="Epochs")

for i in epoch_bar:
    time.sleep(0.01)

Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

### Preprocess Data

In [1]:
from preprocess_data import preprocess_data
from pathlib import Path

print("Preprocessing training data...")
preprocess_data(
    data_dir=Path("Results_large_10_pow_3/"),
    split="test",
    noise_scale=0.0,
    recalc_velocities=True,
)

Preprocessing training data...
Processing file: Results_large_10_pow_3/test/graphs/graphsL0.9_W0.12_D0.04_NL8_NW4_ND4_E1000.0_nu0.3_rho1.0_em0.01_ek0.01_Pix0.0_Piy-1.0_Piz0.3_T4.0_Tc0.8_Nsteps50.pt (1/1)
Processing trajectory: 0
Length of trajectory 0: 49


### Load Dataset

In [ ]:
%%writefile pipeline/load_data.py
from in_memory_dataset import InMemoryTimeStepDataset
from torch_geometric.loader import DataLoader


def get_dataloaders() -> tuple[DataLoader, DataLoader]:
    train_dataset = InMemoryTimeStepDataset(sample_dir="dataset/beam/train")
    test_dataset = InMemoryTimeStepDataset(sample_dir="dataset/beam/val")

    batch_size = 16
    num_workers = 4
    train_dataloader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
    )
    test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    return train_dataloader, test_dataloader

Writing pipeline/load_data.py


### Initialize Model

In [ ]:
%%writefile pipeline/build_model.py


def build_model():
    import torch
    from models.vinay_mgn import MeshGraphNet

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    node_channels = 13
    edge_channels = 8
    num_messages = 8
    latent_dim = 128

    torch.manual_seed(42)
    model = MeshGraphNet(
        node_channels=node_channels,
        edge_channels=edge_channels,
        latent_size=latent_dim,
        num_msgs=num_messages,
    )
    _ = model.to(device)
    return model, device

Writing pipeline/build_model.py


### Initialize Trainer

In [4]:
%%writefile pipeline/setup_trainer.py
import torch
from trainer import Trainer


def setup_trainer(model, device) -> Trainer:
    lr = 5e-5
    loss_type = "mse"

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    trainer = Trainer(model, optimizer, device, loss_type=loss_type)
    torch.cuda.empty_cache()
    print(f"Training_id: {trainer.training_id}")
    return trainer

Writing pipeline/setup_trainer.py


### Train Loop

In [15]:
%%writefile pipeline/train_loop.py

from tqdm import tqdm
from preprocess_data import Stats
import json


def train_loop(
    trainer,
    train_dataloader,
    test_dataloader,
    device,
    total_epochs=50,
):
    print("Loading stats...")
    with open("Results/train/stats/stats.json", "r") as f:
        stats = json.load(f)
    node_stats = Stats.from_dict(stats["node"])
    edge_stats = Stats.from_dict(stats["edge"])
    target_stats = Stats.from_dict(stats["target"])
    dt = 0.08

    epoch_bar = tqdm(range(total_epochs), desc="Epochs")
    for epoch in epoch_bar:
        for batch in tqdm(train_dataloader, desc="Batches", leave=False):
            # print("hey")
            batch.to(device)
            trainer.train(batch)

        trainer.test(
            test_loader=test_dataloader,
            node_stats=node_stats,
            edge_stats=edge_stats,
            target_stats=target_stats,
            dt=dt,
        )
        trainer.epoch_end()
        trainer.save_model(epoch=epoch)
        epoch_bar.set_postfix(
            {
                "last_loss": f"{trainer.loss:.6f}, full_rollout_error: {trainer.rollout_all_step_error:.6f}",
            }
        )
    return trainer

Overwriting pipeline/train_loop.py


In [13]:
%%writefile pipeline/train.py
from preprocess_data_for_training import preprocess_data_pipeline
from load_data import get_dataloaders
from build_model import build_model
from setup_trainer import setup_trainer
from train_loop import train_loop

if __name__ == "__main__":
    preprocess_data_pipeline()
    train_dataloader, test_dataloader = get_dataloaders()
    model, device = build_model()
    trainer = setup_trainer(model, device)
    trainer = train_loop(
        trainer,
        train_dataloader,
        test_dataloader,
        device,
        total_epochs=100,
    )

Overwriting pipeline/train.py


### Show Training Progress

In [5]:
import matplotlib.pyplot as plt
import numpy as np


len(train_dataloader)
epochs_indices = np.arange(
    0, total_epochs * len(train_dataloader), len(train_dataloader)
)
print(epochs_indices)
plt.figure(figsize=(16, 6))
# plt.plot(trainer.train_history, label="Train loss", alpha=0.1, color="gray")
# plt.plot(epochs_indices, trainer.gen_test_history, label="Gen Test Loss")
acc_loss = np.array(trainer.train_acc_history).mean(axis=1)
stress_loss = np.array(trainer.train_stress_history).mean(axis=1)
plt.plot(epochs_indices, acc_loss, label="Acceleration Loss")
plt.plot(epochs_indices, stress_loss, label="Stress Loss")
total_loss = acc_loss + stress_loss
plt.plot(epochs_indices, total_loss, label="Total Loss", linestyle="--", color="black")
plt.legend()
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.xticks(epochs_indices, [str(i) for i in range(total_epochs)])
plt.grid()
plt.show()


NameError: name 'train_dataloader' is not defined

### Show animation

In [ ]:
from rollout_utils import do_rollout

model = trainer.model.eval()
no_rollout = False
true_rollout, pred_rollout = do_rollout(
    model=model,
    test_loader=test_dataloader,
    device=trainer.device,
    node_stats=node_stats,
    edge_stats=edge_stats,
    target_stats=target_stats,
    dt=dt,
    skip_first=0,
    rollout_steps=30,
    dont_rollout=no_rollout,
)

In [ ]:
from make_gif import make_beam_comparison_gif


make_beam_comparison_gif(
    pred_rollout=pred_rollout,
    true_rollout=true_rollout,
    L=1.0,
    W=0.1,
    D=0.04,  # Beam dimensions
    out_gif="beam_comparison.gif",
    fps=4,
)

Rendering frame 0/30
Rendering frame 1/30
Rendering frame 2/30
Rendering frame 3/30
Rendering frame 4/30
Rendering frame 5/30
Rendering frame 6/30
Rendering frame 7/30
Rendering frame 8/30
Rendering frame 9/30
Rendering frame 10/30
Rendering frame 11/30
Rendering frame 12/30
Rendering frame 13/30
Rendering frame 14/30
Rendering frame 15/30
Rendering frame 16/30
Rendering frame 17/30
Rendering frame 18/30
Rendering frame 19/30
Rendering frame 20/30
Rendering frame 21/30
Rendering frame 22/30
Rendering frame 23/30
Rendering frame 24/30
Rendering frame 25/30
Rendering frame 26/30
Rendering frame 27/30
Rendering frame 28/30
Rendering frame 29/30
Rendering frame 30/30
GIF saved => beam_comparison.gif
Done.


In [ ]:
# from IPython.display import Image

# Image(filename="beam_comparison.gif")